### Diffusion of a gaussian packegue

Number density of each flavor component along $x$. Each curve is one snapshot; color is time.

In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import h5py
import numpy as np

try:
    here = Path(__file__).resolve().parent
except NameError:
    here = Path.cwd()

sys.path.append(str((here / '..' / 'nsm_plots').resolve()))
sys.path.append(str((here / '..' / 'phys_const').resolve()))

import plot_functions as pf
import phys_const as pc

Gaussian diffusion test simulation paramters

In [ ]:
# Directory path
directory_path = '/mnt/scratch/erick/erick_scratch/emu_methods_paper_2027/gaussian_diffusion_test'
directory_path = '/mnt/scratch/erick/erick_scratch/emu_methods_paper_2027/gaussian_diffusion_test_smaller_dt'
number_of_flavors = 3

# Simulation domain
ncell = (1024, 1, 1)
Lx = 200.0 # cm
Ly = 0.1953125 # cm
Lz = 0.1953125  # cm
dx = Lx / ncell[0]
dy = Ly / ncell[1]
dz = Lz / ncell[2]
domain = (Lx, Ly, Lz)  # cm
x_cell_centers = np.arange(domain[0] / ncell[0] / 2, domain[0], domain[0] / ncell[0])

# Scattering opacities
kappar_cm = 5.91e-01
kapparbar_cm = 5.91e-01

# Initial conditions of Gaussian packet
# n = n_bg + A0 * exp(-(x - x0)^2 / (2 s0^2))
n_bg_ccm = 1.0e28
A0_ccm = 1.0e29
x0_cm = 100.0
s0_cm = 5.0

# Snapshots drawn on each spatial plot
lowlimfit = None
uplimfit = None

Analitic solution

In [ ]:
att_num_diff = 0.0

D_cm2_s = pc.PhysConst.c / (3.0 * kappar_cm)
D_num_cm2_s = pc.PhysConst.c * kappar_cm * dx**2 / 4.0
D_num_cm2_s *= att_num_diff

print(f'D_cm2_s (physical diffusion) = {D_cm2_s:.2e}')
print(f'D_num_cm2_s (numerical diffusion) = {D_num_cm2_s:.2e}')

Dnum_bigger_D = D_num_cm2_s > D_cm2_s

if Dnum_bigger_D:
    print('numerical diffusion is bigger than physical diffusion!')
    minval = 1.0e-2
    k_ideal  = np.sqrt( 4*minval / (dx     **2 * 3))
    dx_ideal = np.sqrt( 4*minval / (k_ideal**2 * 3))
    print(f"dx (previous): {dx:.2e}, dx_ideal (ideal): {dx_ideal:.2e}")
    print(f"kappar_cm (previous): {kappar_cm:.2e}, k_ideal_cm (ideal): {k_ideal:.2e}")    
else:
    print('physical diffusion is bigger than numerical diffusion!')

D_cm2_s += D_num_cm2_s

def analytic_solution(t_s, x_cm=None):
    """Cartesian reference density n(x, t) in cm^-3.

    n(x, t) = n_bg + A0 * s0 / sqrt(s0^2 + 2 D t)
              * exp(-(x - x0)^2 / (2 (s0^2 + 2 D t)))

    t_s is a time in seconds, or an array of times. x_cm defaults to the
    cell centers of the run. A scalar time returns shape (nx,); an array
    of times returns shape (nt, nx).
    """
    if x_cm is None:
        dx = domain[0] / ncell[0]
        x_cm = np.arange(dx / 2.0, domain[0], dx)
    x_cm = np.asarray(x_cm, dtype=float)
    t_in = np.asarray(t_s, dtype=float)
    variance = s0_cm**2 + 2.0 * D_cm2_s * np.atleast_1d(t_in)
    n = (
        n_bg_ccm
        + A0_ccm * s0_cm / np.sqrt(variance)[:, None]
        * np.exp(-((x_cm - x0_cm) ** 2) / (2.0 * variance[:, None]))
    )
    if t_in.ndim == 0:
        return n[0]
    return n

Reading simulation data

In [ ]:
all_data_path = os.path.join(directory_path, 'allData.h5')
with h5py.File(all_data_path, 'r') as all_data_file:
    nee = np.array(all_data_file['/N00_Re(1|ccm)'][:])
    nuu = np.array(all_data_file['/N11_Re(1|ccm)'][:])
    neu = np.sqrt(
        np.array(all_data_file['/N01_Re(1|ccm)'][:])**2
        + np.array(all_data_file['/N01_Im(1|ccm)'][:])**2
    )
    if number_of_flavors == 3:
        ntt = np.array(all_data_file['/N22_Re(1|ccm)'][:])
        net = np.sqrt(
            np.array(all_data_file['/N02_Re(1|ccm)'][:])**2
            + np.array(all_data_file['/N02_Im(1|ccm)'][:])**2
        )
        nut = np.sqrt(
            np.array(all_data_file['/N12_Re(1|ccm)'][:])**2
            + np.array(all_data_file['/N12_Im(1|ccm)'][:])**2
        )
    time_s = np.array(all_data_file['/t(s)'][:])

Plotting data

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for i in range(len(time_s)):
    analytic_solution_at_time = analytic_solution(time_s[i], x_cell_centers)
    ax.plot(x_cell_centers, nee[i, :, 0, 0],           color="C"+str(i), alpha=0.4, linewidth=3, linestyle="solid")
    ax.plot(x_cell_centers, analytic_solution_at_time, color="C"+str(i), alpha=1.0, linewidth=3, linestyle="dotted")

ax.set_xlabel(r"$x$ [cm]")
ax.set_ylabel(r"$n_{ee}$ [cm$^{-3}$]")
ax.set_xlim(60,140)
ax.legend(fontsize=15, frameon=False)
plt.tight_layout()
plt.show()

pf.plot_colored_lines(
    x_cell_centers,                         # x-axis: space/grid centers, shape (1024,)
    nee[:, :, 0, 0],                      # y values: need shape (space, time)
    time_s,                                 # t-axis: list of times
    r'$z \, [\mathrm{cm}]$',                # x label
    r'$n_{ee} \, [\mathrm{cm}^{-3}$]',      # y label
    r'$t \, [s]$',                          # t label
    f"plots/{directory_path.split('/')[-1]}_n_ee_space.pdf",
    xlog= False, 
    ylog=False, 
    ylowerlimit=None, 
    yupperlimit=None,
    xlowerlimit=100.0,
    xupperlimit=140.0
)


Error stimation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for i in range(len(time_s)):
    analytic_solution_at_time = analytic_solution(time_s[i], x_cell_centers)
    error_estimate_at_time = np.abs(nee[i, :, 0, 0] - analytic_solution_at_time)/analytic_solution_at_time
    ax.plot(x_cell_centers, error_estimate_at_time, color="C"+str(i), alpha=1.0, linewidth=3, linestyle="solid")

    max_error = np.max(error_estimate_at_time)
    max_error_index = np.argmax(error_estimate_at_time)
    max_error_x = x_cell_centers[max_error_index]
    print(f"time {time_s[i]:.2e} s: Maximum relative error: {max_error:.2e} at x = {max_error_x:.2f} cm")

ax.set_yscale("log")
ax.set_xlabel(r"$x$ [cm]")
ax.set_ylabel(r"$|n_{ee} - n_{ee}^{th} | / n_{ee}^{th}$")
ax.set_ylabel(r"Relative error")
ax.legend(fontsize=15, frameon=False)
plt.tight_layout()
plt.show()

error_estimate_at_time = np.zeros_like(nee[:, :, 0, 0])
for i in range(len(time_s)):
    analytic_solution_at_time = analytic_solution(time_s[i], x_cell_centers)
    error_estimate_at_time[i, :] = np.abs(nee[i, :, 0, 0] - analytic_solution_at_time)/analytic_solution_at_time

pf.plot_colored_lines(
    x_cell_centers,                         # x-axis: space/grid centers, shape (1024,)
    np.log10(error_estimate_at_time),                      # y values: need shape (space, time)
    time_s,                                 # t-axis: list of times
    r'$z \, [\mathrm{cm}]$',                # x label
    r'$\log_{10}\,(\,\mathrm{relative\,error}\,)$',      # y label
    r'$t \, [s]$',                          # t label
    f"plots/{directory_path.split('/')[-1]}_n_ee_space.pdf",
    xlog= False, 
    ylog= False, 
    ylowerlimit=None, 
    yupperlimit=None,
    xlowerlimit=100.0,
    xupperlimit=140.0
)


Conservation of number of neutrinos

In [ ]:
number_of_neutrinos = np.zeros_like(time_s)
for i in range(len(time_s)):
    number_of_neutrinos[i] = np.sum(nee[i, :, 0, 0]*dx*dy*dz)

plt.figure(figsize=(8,6))
plt.plot(time_s, number_of_neutrinos/number_of_neutrinos[0] - 1, linewidth=2)
plt.xlabel(r'$t \, [s]$')
plt.ylabel(r'$N_{\nu}/N_{\nu}(t=0) - 1$')
# plt.yscale("log")
plt.title("Total Number of Neutrinos vs Time")
plt.tight_layout()
plt.savefig(f"plots/{directory_path.split('/')[-1]}_n_nu_time.pdf")
plt.show()

